# 01b — Phân tích lại G1 trên CPU (tái tạo số trong sổ §10)

Notebook này **không cần GPU, không tải model**. Nó đọc `results/<run_id>/` của G1 và:

1. kiểm dữ liệu từng tác vụ (split, trùng train/eval, lệch vị trí base/source);
2. tổng hợp odds và selectivity **đúng cách bảng của bài gốc** (`plot.py:322`, `plot.py:328–330`): max theo vùng ở mỗi layer → trung bình qua layer;
3. tính lại **độc lập** 3 tiêu chí G1 (sổ §4, D-13), không dùng `g1_check.json`, rồi so với `g1_check.json`;
4. tách phần lệch so với bài gốc theo nhóm tác vụ có / không có lệch vị trí (sổ §8 điểm 9);
5. nếu có kết quả chạy `POS_MODE = "repo"` (G1r), áp quy tắc **D-16** đã đặt trước khi chạy.

Mọi con số G1 trong sổ dự án phải in ra được từ notebook này (QT-32).
Chạy: máy cá nhân hoặc Kaggle CPU, Run All. `TOY_TEST = True` dùng dữ liệu giả để kiểm code.

In [1]:
# parameters (papermill). Comment không có dấu bằng (QT-21).
# Thư mục kết quả G1 (POS_MODE correct), tính từ gốc repo
RUN_DIR = "results/20261113_0846_g1"
JOBS = ["g1_410m", "g1_1b"]
# Thư mục kết quả G1r (POS_MODE repo); để rỗng nếu chưa chạy
REPO_RUN_DIR = ""
REPO_JOBS = ["g1r_410m", "g1r_1b"]
OUT_DIR = "outputs/01b_phan_tich_g1"
TOY_TEST = False

In [2]:
# Parameters
OUT_DIR = "results/20261116_claude_cpu_nb01b"


## Ô 1. Phiên bản và thư mục ghi (QT-26)

In [3]:
import os, sys, json, glob, math, platform
import numpy as np
import pandas as pd
os.makedirs(OUT_DIR, exist_ok=True)
OUT_JSONL = os.path.join(OUT_DIR, "phan_tich_g1.jsonl")
open(OUT_JSONL, "w").close()
def ghi(rec):
    with open(OUT_JSONL, "a", encoding="utf-8") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
print({"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__, "toy": TOY_TEST})
pd.set_option("display.width", 200)

{'python': '3.13.18', 'numpy': '2.5.3', 'pandas': '3.0.5', 'toy': False}


## Ô 2. Số của bài gốc

Chép từ PDF arXiv 2402.12560v1 (bản trong Project), **Table 11** (410m, odds), **Table 12** (410m, selectivity), **Table 13** (1b, odds), **Table 14** (1b, selectivity). Claude đối chiếu từng số với PDF ngày 16/11/2026; số trong `notebooks/01_tai_lap_causalgym.ipynb` trùng các số này.
Thứ tự cột: DAS, Probe0, Probe1, Mean, PCA, k-means, LDA, Rand., Vanilla; cột cuối của odds là Task Acc.

In [4]:
COLS = ["das", "probe0", "probe1", "mean", "pca", "kmeans", "lda", "random", "vanilla"]
TASKS = ["agr_sv_num_subj-relc", "npi_any_subj-relc", "filler_gap_subj", "cleft"]
_P = {
 "410m": {"odds": {
   "agr_sv_num_subj-relc": [4.97, 4.32, 4.19, 3.72, 0.29, 0.28, 1.88, 0.03, 4.42, 0.97],
   "npi_any_subj-relc":    [8.76, 4.01, 4.00, 3.99, 2.28, 2.34, 0.15, 0.01, 4.08, 0.95],
   "filler_gap_subj":      [11.84, 6.53, 6.44, 6.41, 4.87, 4.98, 0.14, 0.01, 6.34, 0.89],
   "cleft":                [14.46, 5.53, 4.84, 1.78, 0.86, 0.92, 0.05, 0.02, 5.79, 0.95]},
  "sel": {
   "agr_sv_num_subj-relc": [3.64, 5.63, 5.45, 4.84, 0.40, 0.38, 2.43, 0.03, 5.52],
   "npi_any_subj-relc":    [2.13, 4.13, 4.12, 4.12, 2.31, 2.39, 0.14, 0.01, 4.26],
   "filler_gap_subj":      [4.23, 6.28, 6.04, 5.98, 4.05, 4.18, 0.15, 0.01, 5.81],
   "cleft":                [6.91, 12.09, 10.55, 3.99, 2.02, 2.13, 0.08, 0.02, 12.91]}},
 "1b": {"odds": {
   "agr_sv_num_subj-relc": [6.62, 5.23, 4.98, 4.24, 0.36, 0.39, 1.40, 0.01, 5.51, 1.00],
   "npi_any_subj-relc":    [10.51, 4.33, 4.33, 4.33, 2.80, 2.93, 0.20, 0.00, 4.41, 0.97],
   "filler_gap_subj":      [11.82, 6.23, 6.18, 6.17, 3.76, 3.87, 0.03, 0.00, 6.22, 0.89],
   "cleft":                [15.56, 5.59, 3.76, 1.62, 0.25, 0.43, 0.16, 0.01, 6.04, 0.97]},
  "sel": {
   "agr_sv_num_subj-relc": [4.13, 6.97, 6.59, 5.60, 0.42, 0.46, 1.75, 0.01, 7.32],
   "npi_any_subj-relc":    [3.32, 4.63, 4.62, 4.62, 2.95, 3.08, 0.21, 0.01, 4.83],
   "filler_gap_subj":      [3.51, 5.94, 5.82, 5.80, 2.96, 3.25, 0.03, 0.01, 5.74],
   "cleft":                [4.05, 11.38, 7.64, 3.36, 0.58, 0.96, 0.33, 0.03, 12.31]}},
}
PAPER = {mk: {"odds": {t: dict(zip(COLS, v[:9])) for t, v in d["odds"].items()},
              "acc": {t: v[9] for t, v in d["odds"].items()},
              "sel": {t: dict(zip(COLS, v)) for t, v in d["sel"].items()}} for mk, d in _P.items()}
print("410m agr odds:", PAPER["410m"]["odds"]["agr_sv_num_subj-relc"])

410m agr odds: {'das': 4.97, 'probe0': 4.32, 'probe1': 4.19, 'mean': 3.72, 'pca': 0.29, 'kmeans': 0.28, 'lda': 1.88, 'random': 0.03, 'vanilla': 4.42}


## Ô 3. Dữ liệu giả (chỉ khi `TOY_TEST = True`)

Sinh `results.jsonl` giả cho 2 model × 4 tác vụ (và G1r giả cho 2 tác vụ), cùng cấu trúc dòng với notebook 01 (`meta`, `data_check`, `behavior`, `site`). Số giả không có ý nghĩa; chỉ để chạy hết code.

In [5]:
def _toy_run(root, jobs, tasks, pos_mode, scale_bug):
    rng = np.random.default_rng(0)
    for job in jobs:
        mk = "1b" if "1b" in job else "410m"
        os.makedirs(os.path.join(root, job), exist_ok=True)
        L = 4; regions = [1, 2]
        rows = [dict(exp="meta", model=f"EleutherAI/pythia-{mk}", pos_mode=pos_mode, eval_split="test", toy=True)]
        for t in tasks:
            mis = 0.1 if t in ("agr_sv_num_subj-relc", "cleft") else 0.0
            rows.append(dict(exp="data_check", task=t, n_train=40, n_eval=20, tok_mismatch=0, n_checked=120,
                             eval_sentences_in_train=0, n_eval_sentences=20, eval_swap_frac=1.0, pos_mismatch_frac=mis,
                             first_token_is_eos=1.0, multi_token_labels=0, diff_region=1, regions=regions))
            rows.append(dict(exp="behavior", task=t, acc=PAPER[mk]["acc"][t], logit_diff=3.0, n_pairs=20,
                             paper_acc=PAPER[mk]["acc"][t]))
            for m in COLS:
                for manip in ("none", "dog-give"):
                    for l in range(L):
                        for r in regions:
                            base = PAPER[mk]["odds"][t][m] if manip == "none" else 0.1
                            if mis and m in ("mean", "pca", "kmeans") and manip == "none":
                                base *= scale_bug
                            ramp = 1.0 if (r == 1 or l >= 2) else 0.2
                            rows.append(dict(exp="site", task=t, layer=l, region=r, method=m, manip=manip,
                                             odds=float(base * ramp * (1 + 0.02 * rng.standard_normal()))))
        with open(os.path.join(root, job, "results.jsonl"), "w") as f:
            for x in rows: f.write(json.dumps(x) + "\n")

if TOY_TEST:
    RUN_DIR = os.path.join(OUT_DIR, "toy_g1"); REPO_RUN_DIR = os.path.join(OUT_DIR, "toy_g1r")
    JOBS = ["g1_410m", "g1_1b"]; REPO_JOBS = ["g1r_410m", "g1r_1b"]
    _toy_run(RUN_DIR, JOBS, TASKS, "correct", 3.0)
    _toy_run(REPO_RUN_DIR, REPO_JOBS, ["agr_sv_num_subj-relc", "cleft"], "repo", 1.0)
    print("Đã sinh dữ liệu giả ở", RUN_DIR, "và", REPO_RUN_DIR)

## Ô 4. Đọc kết quả và kiểm dữ liệu

Mỗi job một model. In: split eval, số cặp, `tok_mismatch`, câu eval trùng train (đếm câu khác nhau), tỉ lệ cặp có vị trí base ≠ source (`pos_mismatch_frac`; chỉ ở các cặp này `POS_MODE` mới làm kết quả khác), accuracy và logit diff so với bài gốc.

In [6]:
def load_job(run_dir, job):
    rows = [json.loads(l) for l in open(os.path.join(run_dir, job, "results.jsonl"), encoding="utf-8")]
    meta = [r for r in rows if r["exp"] == "meta"][0]
    mk = "1b" if "1b" in meta["model"] else "410m"
    return dict(job=job, mk=mk, meta=meta, rows=rows)

def data_table(J):
    dc = {r["task"]: r for r in J["rows"] if r["exp"] == "data_check"}
    bh = {r["task"]: r for r in J["rows"] if r["exp"] == "behavior"}
    out = []
    for t in dc:
        d = dc[t]; b = bh.get(t, {})
        out.append(dict(model=J["mk"], task=t, split=J["meta"].get("eval_split"), n_eval=d["n_eval"],
                        tok_mismatch=f"{d['tok_mismatch']}/{d['n_checked']}",
                        eval_in_train=f"{d['eval_sentences_in_train']}/{d['n_eval_sentences']}",
                        pos_mismatch=d["pos_mismatch_frac"], acc=b.get("acc"), paper_acc=b.get("paper_acc"),
                        logit_diff=round(b.get("logit_diff", float("nan")), 2)))
    return pd.DataFrame(out)

G1 = {j: load_job(RUN_DIR, j) for j in JOBS if os.path.exists(os.path.join(RUN_DIR, j, "results.jsonl"))}
print("Job đọc được:", list(G1), "| pos_mode:", {j: J["meta"]["pos_mode"] for j, J in G1.items()})
DT = pd.concat([data_table(J) for J in G1.values()], ignore_index=True)
print(DT.to_string(index=False))
for _, r in DT.iterrows():
    ghi(dict(buoc="data_check", **{k: (v.item() if hasattr(v, "item") else v) for k, v in r.items()}))

Job đọc được: ['g1_410m', 'g1_1b'] | pos_mode: {'g1_410m': 'correct', 'g1_1b': 'correct'}
model                 task split  n_eval tok_mismatch eval_in_train  pos_mismatch  acc  paper_acc  logit_diff
 410m agr_sv_num_subj-relc  test     100       0/1000          4/99          0.16 0.97       0.97        3.67
 410m    npi_any_subj-relc  test     100       0/1000         0/100          0.00 0.95       0.95        3.09
 410m      filler_gap_subj  test     100       0/1000         0/100          0.00 0.89       0.89        4.00
 410m                cleft  test     100       0/1000         21/70          0.08 0.95       0.95        3.30
   1b agr_sv_num_subj-relc  test     100       0/1000          4/99          0.16 1.00       1.00        4.38
   1b    npi_any_subj-relc  test     100       0/1000         0/100          0.00 0.97       0.97        3.40
   1b      filler_gap_subj  test     100       0/1000         0/100          0.00 0.89       0.89        4.16
   1b                cleft  te

## Ô 5. Tổng hợp kiểu bảng của bài gốc

- **odds** của (tác vụ, phương pháp) = trung bình qua layer của (max theo vùng của odds) — `plot.py:322`, `plot.py:328–330`.
- **selectivity** = trung bình qua layer của (max theo vùng của odds tác vụ gốc − odds dog-give ở cùng ô) — sổ §8 điểm 7.

In [7]:
def aggregate(J):
    site = {}
    for r in J["rows"]:
        if r["exp"] == "site":
            site.setdefault((r["task"], r["manip"], r["method"]), {})[(r["layer"], r["region"])] = r["odds"]
    tasks = sorted({k[0] for k in site}, key=lambda t: TASKS.index(t) if t in TASKS else 99)
    odds, sel = {}, {}
    for t in tasks:
        odds[t], sel[t] = {}, {}
        for m in COLS:
            dn = site.get((t, "none", m)); dc = site.get((t, "dog-give", m))
            if not dn: continue
            per = {}
            for (l, rg), v in dn.items(): per[l] = max(per.get(l, -1e9), v)
            odds[t][m] = float(np.mean(list(per.values())))
            if dc:
                per = {}
                for (l, rg), v in dn.items(): per[l] = max(per.get(l, -1e9), v - dc[(l, rg)])
                sel[t][m] = float(np.mean(list(per.values())))
    return dict(site=site, tasks=tasks, odds=odds, sel=sel)

AG = {j: aggregate(J) for j, J in G1.items()}
for j, A in AG.items():
    print(f"== {j}: odds (hàng = tác vụ)")
    print(pd.DataFrame(A["odds"]).T[COLS].round(2).to_string())

== g1_410m: odds (hàng = tác vụ)
                        das  probe0  probe1  mean   pca  kmeans   lda  random  vanilla
agr_sv_num_subj-relc   5.55    4.63    4.59  4.58  0.98    1.02  0.16    0.00     4.75
npi_any_subj-relc      8.07    4.01    4.00  3.99  2.28    2.34  0.15    0.00     4.08
filler_gap_subj       11.51    6.53    6.44  6.41  4.87    4.98  0.18    0.01     6.34
cleft                 14.29    5.52    5.49  5.48  5.34    5.48  0.17    0.01     5.93
== g1_1b: odds (hàng = tác vụ)
                        das  probe0  probe1  mean   pca  kmeans   lda  random  vanilla
agr_sv_num_subj-relc   7.05    5.58    5.57  5.57  2.66    2.82  0.26     0.0     5.84
npi_any_subj-relc      9.76    4.33    4.33  4.33  2.80    2.93  0.20     0.0     4.41
filler_gap_subj       11.44    6.23    6.19  6.17  3.77    3.88  0.19     0.0     6.23
cleft                 15.89    5.81    5.79  5.78  5.65    5.78  0.29     0.0     6.13


## Ô 6. Ba tiêu chí G1 (sổ §4, D-13), tính độc lập

1. Kendall τ_b giữa odds trung bình 4 tác vụ của mình và của bài (9 phương pháp) ≥ 0,7, và DAS đứng đầu.
2. Layer xuất hiện (layer đầu tiên có odds DAS ≥ 0,5 × max của vùng đó) của vùng khác nhau đầu tiên ≤ của vùng cuối ở ≥ 3/4 tác vụ.
3. ≥ 12/16 cặp (tác vụ × {DAS, probe tốt hơn, diff-in-means, vanilla}) có tỉ lệ mình/bài trong [0,5; 2].

Kiểm phụ: accuracy lệch ≤ 0,05. Cuối ô so với `g1_check.json` do notebook 01 ghi (nếu có).

In [8]:
def kendall_tau_b(x, y):
    n = len(x); nc = nd = tx = ty = 0
    for i in range(n):
        for k in range(i + 1, n):
            a = np.sign(x[i] - x[k]); b = np.sign(y[i] - y[k])
            if a == 0 and b == 0: continue
            if a == 0: tx += 1
            elif b == 0: ty += 1
            elif a == b: nc += 1
            else: nd += 1
    den = math.sqrt((nc + nd + tx) * (nc + nd + ty))
    return (nc - nd) / den if den else float("nan")

def arrival(site, t):
    das = site[(t, "none", "das")]
    regs = sorted({k[1] for k in das}); out = {}
    for rg in regs:
        lv = {l: v for (l, r_), v in das.items() if r_ == rg}
        mx = max(lv.values())
        out[rg] = min(l for l, v in lv.items() if v >= 0.5 * mx)
    return regs[0], regs[-1], out

def g1_criteria(J, A):
    mk = J["mk"]; P = PAPER[mk]; tasks = [t for t in A["tasks"] if t in P["odds"]]
    ours = [float(np.mean([A["odds"][t][m] for t in tasks])) for m in COLS]
    pap = [float(np.mean([P["odds"][t][m] for t in tasks])) for m in COLS]
    tau = kendall_tau_b(ours, pap); das_top = COLS[int(np.argmax(ours))] == "das"
    pos = []
    for t in tasks:
        f, l, arr = arrival(A["site"], t)
        pos.append(dict(task=t, first=f, last=l, arr_first=arr[f], arr_last=arr[l], ok=arr[f] <= arr[l]))
    mags = []
    for t in tasks:
        for m in ("das", "probe", "mean", "vanilla"):
            if m == "probe":
                o = max(A["odds"][t]["probe0"], A["odds"][t]["probe1"]); p = max(P["odds"][t]["probe0"], P["odds"][t]["probe1"])
            else:
                o, p = A["odds"][t][m], P["odds"][t][m]
            mags.append(dict(task=t, method=m, ours=o, paper=p, ratio=o / p, within=0.5 <= o / p <= 2))
    acc = {r["task"]: abs(r["acc"] - P["acc"][r["task"]]) for r in J["rows"] if r["exp"] == "behavior"}
    res = dict(model=mk, n_tasks=len(tasks), tau_b=tau, das_top=das_top, tc1=tau >= 0.7 and das_top,
               n_pos_ok=sum(p["ok"] for p in pos), tc2=sum(p["ok"] for p in pos) >= 3,
               n_within=sum(x["within"] for x in mags), n_pairs=len(mags), tc3=sum(x["within"] for x in mags) >= 12,
               max_acc_diff=max(acc.values()))
    res["g1_dat"] = bool(res["tc1"] and res["tc2"] and res["tc3"] and len(tasks) == 4)
    return res, ours, pap, pos, mags

CRIT = {}
for j, J in G1.items():
    res, ours, pap, pos, mags = g1_criteria(J, AG[j]); CRIT[j] = res
    print(f"\n===== {j} ({res['model']})")
    print("TC1  " + " ".join(f"{m}:{a:.2f}/{b:.2f}" for m, a, b in zip(COLS, ours, pap)))
    print(f"     tau_b = {res['tau_b']:.3f}, DAS đầu = {res['das_top']} -> {'ĐẠT' if res['tc1'] else 'KHÔNG'}")
    for p in pos:
        print(f"TC2  {p['task']:22s} vùng {p['first']}: L{p['arr_first']}  vùng {p['last']}: L{p['arr_last']}  {'ok' if p['ok'] else 'KHÔNG'}")
    print(f"     {res['n_pos_ok']}/{len(pos)} -> {'ĐẠT' if res['tc2'] else 'KHÔNG'}")
    for x in mags:
        if not x["within"]:
            print(f"TC3  ngoài khoảng: {x['task']} {x['method']} {x['ours']:.2f}/{x['paper']:.2f} = {x['ratio']:.2f}")
    print(f"     {res['n_within']}/{res['n_pairs']} -> {'ĐẠT' if res['tc3'] else 'KHÔNG'}")
    print(f"Accuracy lệch lớn nhất: {res['max_acc_diff']:.2f}")
    print(f"==> G1 {'ĐẠT' if res['g1_dat'] else 'CHƯA ĐẠT'} cho {res['model']}")
    ghi(dict(buoc="g1_tieu_chi", job=j, **res, das_ratio={x["task"]: round(x["ratio"], 3) for x in mags if x["method"] == "das"}))
    chk = os.path.join(RUN_DIR, j, "g1_check.json")
    if os.path.exists(chk):
        g = json.load(open(chk, encoding="utf-8"))
        same = (abs(g["order"]["kendall_tau"] - res["tau_b"]) < 1e-9 and g["order"]["das_top"] == res["das_top"]
                and g["magnitude"]["n_within"] == res["n_within"] and sum(p["ok"] for p in g["position"]) == res["n_pos_ok"])
        print("So với g1_check.json của notebook 01:", "TRÙNG" if same else "KHÁC, xem lại")
        ghi(dict(buoc="so_g1_check", job=j, trung=bool(same)))


===== g1_410m (410m)
TC1  das:9.86/10.01 probe0:5.17/5.10 probe1:5.13/4.87 mean:5.12/3.98 pca:3.37/2.07 kmeans:3.45/2.13 lda:0.16/0.55 random:0.01/0.02 vanilla:5.27/5.16
     tau_b = 1.000, DAS đầu = True -> ĐẠT
TC2  agr_sv_num_subj-relc   vùng 1: L0  vùng 5: L17  ok
TC2  npi_any_subj-relc      vùng 1: L0  vùng 8: L13  ok
TC2  filler_gap_subj        vùng 2: L0  vùng 8: L15  ok
TC2  cleft                  vùng 2: L0  vùng 3: L6  ok
     4/4 -> ĐẠT
TC3  ngoài khoảng: cleft mean 5.48/1.78 = 3.08
     15/16 -> ĐẠT
Accuracy lệch lớn nhất: 0.00
==> G1 ĐẠT cho 410m
So với g1_check.json của notebook 01: TRÙNG

===== g1_1b (1b)
TC1  das:11.03/11.13 probe0:5.49/5.35 probe1:5.47/4.81 mean:5.46/4.09 pca:3.72/1.79 kmeans:3.85/1.91 lda:0.24/0.45 random:0.00/0.01 vanilla:5.65/5.54
     tau_b = 1.000, DAS đầu = True -> ĐẠT
TC2  agr_sv_num_subj-relc   vùng 1: L0  vùng 5: L13  ok
TC2  npi_any_subj-relc      vùng 1: L0  vùng 8: L9  ok
TC2  filler_gap_subj        vùng 2: L0  vùng 8: L12  ok
TC2  cleft   

## Ô 7. Phần lệch so với bài gốc, tách theo lệch vị trí

Nếu pipeline của nhóm giống pipeline của bài ở mọi chỗ trừ vị trí lấy source (sổ §8 điểm 9), thì ở tác vụ **không** có cặp lệch vị trí (`pos_mismatch = 0`), các phương pháp tất định (probe0, probe1, mean, PCA, k-means, vanilla) phải cho tỉ lệ mình/bài ≈ 1. Phần lệch phải dồn vào tác vụ có cặp lệch vị trí. DAS và LDA không tất định giữa hai lần chạy nên in riêng.
Cuối ô in tỉ lệ DAS/vanilla (đối chiếu Tanyel & Ayvaz: DAS > vanilla, trung vị khoảng 2).

In [9]:
DET = ["probe0", "probe1", "mean", "pca", "kmeans", "vanilla"]
for j, J in G1.items():
    A = AG[j]; P = PAPER[J["mk"]]
    mis = {r["task"]: r["pos_mismatch_frac"] for r in J["rows"] if r["exp"] == "data_check"}
    R = pd.DataFrame({t: {m: (A["odds"][t][m] / P["odds"][t][m] if P["odds"][t][m] > 0 else np.nan) for m in COLS} for t in A["tasks"]}).T[COLS]
    S = pd.DataFrame({t: {m: A["sel"][t][m] - P["sel"][t][m] for m in COLS} for t in A["tasks"]}).T[COLS]
    print(f"\n===== {j}: tỉ lệ odds mình/bài")
    print(R.round(2).to_string())
    print("Hiệu selectivity mình − bài:")
    print(S.round(2).to_string())
    for grp, ts in (("không lệch vị trí", [t for t in A["tasks"] if mis.get(t, 0) == 0]),
                    ("có lệch vị trí", [t for t in A["tasks"] if mis.get(t, 0) > 0])):
        vals = R.loc[ts, DET].values.ravel()
        vals = vals[~np.isnan(vals)]
        if len(vals):
            dev = np.max(np.abs(vals - 1))
            print(f"{grp:18s} {ts}: |tỉ lệ − 1| lớn nhất của phương pháp tất định = {dev:.3f}")
            ghi(dict(buoc="lech_theo_nhom", job=j, nhom=grp, tasks=ts, max_abs_dev=float(dev)))
    dv = {t: A["odds"][t]["das"] / A["odds"][t]["vanilla"] for t in A["tasks"]}
    print("DAS/vanilla:", {t: round(v, 2) for t, v in dv.items()}, "| trung vị", round(float(np.median(list(dv.values()))), 2))
    ghi(dict(buoc="das_vanilla", job=j, ratio={t: float(v) for t, v in dv.items()}))
    ghi(dict(buoc="ti_le", job=j, odds_ratio=R.round(4).to_dict(orient="index"), sel_diff=S.round(4).to_dict(orient="index")))


===== g1_410m: tỉ lệ odds mình/bài
                       das  probe0  probe1  mean   pca  kmeans   lda  random  vanilla
agr_sv_num_subj-relc  1.12    1.07    1.09  1.23  3.37    3.65  0.09    0.12     1.08
npi_any_subj-relc     0.92    1.00    1.00  1.00  1.00    1.00  1.03    0.38     1.00
filler_gap_subj       0.97    1.00    1.00  1.00  1.00    1.00  1.26    0.85     1.00
cleft                 0.99    1.00    1.13  3.08  6.20    5.96  3.30    0.34     1.02
Hiệu selectivity mình − bài:
                       das  probe0  probe1  mean   pca  kmeans   lda  random  vanilla
agr_sv_num_subj-relc  0.39    0.45    0.57  1.17  0.91    1.01 -2.21   -0.02     0.57
npi_any_subj-relc     0.46   -0.00   -0.00 -0.00  0.00   -0.00  0.00   -0.01     0.00
filler_gap_subj       0.36    0.00   -0.00  0.00 -0.00   -0.00  0.03    0.00     0.00
cleft                -0.41    0.18    1.67  8.21  9.92   10.07  0.22   -0.01     0.36
không lệch vị trí  ['npi_any_subj-relc', 'filler_gap_subj']: |tỉ lệ − 1| lớ

## Ô 8. So chế độ `repo` (G1r) — quy tắc D-16, đặt trước khi chạy (16/11/2026)

Giả thuyết: lệch ở `agr_sv_num_subj-relc` và `cleft` là do lỗi vị trí của repo gốc (sổ §8 điểm 9).
- 12 ô kiểm: {mean, PCA, k-means} × {`agr_sv_num_subj-relc`, `cleft`} × {410m, 1b}. Ở chế độ `correct` (G1) có 2/12 ô có tỉ lệ mình/bài trong [0,5; 2].
- Ở chế độ `repo`: **≥ 10/12** ô trong [0,5; 2] → *Xác nhận*; **7–9** → *Giải thích một phần*; **≤ 6** → *Không giải thích được*. Thiếu ô (job lỗi hoặc bị bỏ) → *Chưa đủ dữ liệu*.
- In thêm (không dùng để kết luận): số ô trong [0,9; 1,1] (mức khớp như ở hai tác vụ không lệch vị trí), và tỉ lệ của DAS, vanilla, probe, LDA.

Ô này bỏ qua nếu `REPO_RUN_DIR` rỗng.

In [10]:
D16_TASKS = ["agr_sv_num_subj-relc", "cleft"]; D16_METHODS = ["mean", "pca", "kmeans"]
def d16_count(AGx, Jx):
    cells = []
    for j, A in AGx.items():
        mk = Jx[j]["mk"]
        for t in D16_TASKS:
            for m in D16_METHODS:
                if t in A["odds"] and m in A["odds"][t]:
                    cells.append(dict(model=mk, task=t, method=m, ratio=A["odds"][t][m] / PAPER[mk]["odds"][t][m]))
    return cells

c_cor = d16_count(AG, G1)
print("Chế độ correct (G1):", sum(0.5 <= c["ratio"] <= 2 for c in c_cor), "/", len(c_cor), "ô trong [0,5; 2]")
if REPO_RUN_DIR:
    GR = {j: load_job(REPO_RUN_DIR, j) for j in REPO_JOBS if os.path.exists(os.path.join(REPO_RUN_DIR, j, "results.jsonl"))}
    for j, J in GR.items():
        assert J["meta"]["pos_mode"] == "repo", f"{j} không chạy ở POS_MODE repo"
    AGR = {j: aggregate(J) for j, J in GR.items()}
    print(pd.concat([data_table(J) for J in GR.values()], ignore_index=True).to_string(index=False))
    c_rep = d16_count(AGR, GR)
    n_in = sum(0.5 <= c["ratio"] <= 2 for c in c_rep); n_close = sum(0.9 <= c["ratio"] <= 1.1 for c in c_rep)
    if len(c_rep) < 12: nhan = "Chưa đủ dữ liệu"
    elif n_in >= 10: nhan = "Xác nhận"
    elif n_in >= 7: nhan = "Giải thích một phần"
    else: nhan = "Không giải thích được"
    print(f"Chế độ repo (G1r): {n_in}/{len(c_rep)} ô trong [0,5; 2]; {n_close}/{len(c_rep)} ô trong [0,9; 1,1] -> D-16: {nhan}")
    rows = []
    for j, A in AGR.items():
        mk = GR[j]["mk"]; jc = [k for k, v in G1.items() if v["mk"] == mk]
        for t in A["tasks"]:
            for m in COLS:
                p = PAPER[mk]["odds"][t][m]
                cor = AG[jc[0]]["odds"][t][m] if jc else np.nan
                rows.append(dict(model=mk, task=t, method=m, paper=p, repo=A["odds"][t][m], correct=cor,
                                 repo_ratio=A["odds"][t][m] / p if p else np.nan, correct_ratio=cor / p if p else np.nan))
    TR = pd.DataFrame(rows)
    print(TR.round(2).to_string(index=False))
    ghi(dict(buoc="d16", n_in=n_in, n_close=n_close, n_cells=len(c_rep), nhan=nhan, cells=c_rep))
    ghi(dict(buoc="g1r_bang", rows=TR.round(4).to_dict(orient="records")))
else:
    print("Chưa có REPO_RUN_DIR: bỏ qua D-16.")

Chế độ correct (G1): 2 / 12 ô trong [0,5; 2]
Chưa có REPO_RUN_DIR: bỏ qua D-16.


## Ô 9. Kết luận ghi file

`phan_tich_g1.jsonl` trong `OUT_DIR` có mọi số in ở trên (mỗi bước một dòng). Quyết định G1 do nhóm theo D-13; notebook chỉ in số.

In [11]:
print(json.dumps({j: {k: CRIT[j][k] for k in ("model", "tau_b", "n_pos_ok", "n_within", "g1_dat")} for j in CRIT}, ensure_ascii=False, indent=1))
print("Số dòng ghi:", sum(1 for _ in open(OUT_JSONL, encoding="utf-8")), "->", OUT_JSONL)

{
 "g1_410m": {
  "model": "410m",
  "tau_b": 1.0,
  "n_pos_ok": 4,
  "n_within": 15,
  "g1_dat": true
 },
 "g1_1b": {
  "model": "1b",
  "tau_b": 1.0,
  "n_pos_ok": 4,
  "n_within": 15,
  "g1_dat": true
 }
}
Số dòng ghi: 20 -> results/20261116_claude_cpu_nb01b/phan_tich_g1.jsonl
